# 2. Data Inspection

Before we start doing anything with the data, we want to have a look at them! 
In this notebook we will inspect various aspects of the data. 

- Plot EEG data
- compute noise covariance
- compute data covariance


<div class="alert alert-success">
    <b>Goal of this notebook</b>:
     <ul>
      <li>Load EEG data (epoched)</li>
      <li>Inspect epoch metadata and understand their structure</li>
      <li>Plot ERP traces, and different experimental conditions</li>
    </ul>
</div>

In [ ]:
# imports
import os
import mne
import numpy as np
import matplotlib.pyplot as plt

# data paths
subID = 24
data_path = "data"
subject_path = os.path.join(f"sub-0{subID}", "ses-mecha", "eeg")
epoch_file = f"sub-0{subID}_ses-mecha_task-NT_epo.fif"
epochs_path = os.path.join("..", data_path, subject_path, epoch_file)

### 2.1 Load EEG Data
We load and inspect data from one participant.

In [ ]:
# load data
epochs = mne.read_epochs(epochs_path, preload=True)
# crop for visualization
epochs.crop(tmin=-0.1, tmax=0.3)

# inspect EEG data object
epochs

### 2.2 What is in the epochs object?

The epochs are cut around the onset of a near-threshold mechanical stimulus.
Every trial carries metadata, most importantly `response_decoded` (did the
participant report feeling the stimulus?) and `intensity_level` (how strong was the stimulus?).

<div class="alert alert-warning">
    <b>Exercise</b>:
    Take a closer look at the metatdata and the conditions in the experiment.
     <ul>
      <li>How many trials does this dataset have?</li>
      <li>How many trials were perceived and how many were not perceived?</li>
      <li>How many intensity levels do we have?</li>
    </ul>
</div>

... double click to type your answer ...

In [ ]:
# the metadata table: one row per trial
epochs.metadata.head()

In [ ]:
# trials per condition
print(epochs.metadata["response_decoded"].value_counts(), "\n")

# trials per stimulus intensity and response -> the psychometric structure
print(epochs.metadata.groupby(["intensity_level", "response_decoded"]).size().unstack())

In [ ]:
# with metadata, trials are selected by writing pandas queries
perceived = epochs["response_decoded == 'Yes'"]
unperceived = epochs["response_decoded == 'No'"]
print(perceived)
print(unperceived)

### 2.3 EEG electrode layout

In [ ]:
# this montage defines the forward model later on in the source reconstruction process
# remember: the forward model tells us what the sensors would record if a source would be active
epochs.plot_sensors(show_names=True, sphere="auto")
plt.show()

### 2.4 Spectrum

Sanity check for line noise, drifts and bad channels: power spectrum per
channel and the topography of the classical frequency bands.

In [ ]:
spectrum = epochs.compute_psd(fmax=45)

# one line per channel - outliers are candidates for bad channels
spectrum.plot(picks="eeg", amplitude=False, spatial_colors=True)
plt.show()

### 2.5 Plotting single trials

`plot_image` shows every trial as one row of a colour-coded image, with the
average underneath. Sorting the rows by metadata makes condition differences
visible in the single-trial data.

<div class="alert alert-warning">
    <b>Exercise</b>:
    The plot below shows each epoch timecourse for a certain electrode and the averaged epochs in gray below.
    <ul>
      <li>Try out plotting different electrodes.</li>
      <li>Where would we expect the stronges elicited activity? Find an electrode with a clear response after stimulus onset.</li>
    </ul>
</div>

... double click to type your answer ...

In [ ]:
roi = ["C6"]

epochs.plot_image(picks=roi, combine="mean", sigma=1.0, title="ROI mean")
plt.show()

### 2.6 Perceived vs. unperceived - Evoked response
Is there a difference in response between perceived and unperceived trials?
Later we will apply this contrast to the source reconstruction too.

In [ ]:
evokeds = {"perceived": perceived.average(), "unperceived": unperceived.average()}
print("ROIs", roi)
mne.viz.plot_compare_evokeds(evokeds, picks=roi, combine="mean",
                             show_sensors="upper right", title="ROI mean")
plt.show()

<div class="alert alert-danger">
    <b>Warning</b>:
    We are currently only looking at one single participant. In a real analysis, for solid interpretation and effects, we need many more!
    Below you can see that the confidence interval largely overlap between the conditions.
</div>

In [ ]:
# with confidence intervals: pass the single-trial evokeds instead of the averages
evokeds_trials = {"perceived": list(perceived.copy().pick(roi).iter_evoked()),
                  "unperceived": list(unperceived.copy().pick(roi).iter_evoked())}

mne.viz.plot_compare_evokeds(evokeds_trials, picks=roi, combine="mean", ci=0.95,
                             show_sensors="upper right", title="ROI mean, 95% CI")
plt.show()

In [ ]:
# difference wave
difference = mne.combine_evoked([evokeds["perceived"], evokeds["unperceived"]],
                                weights=[1, -1])
difference.comment = "perceived - unperceived"
difference.plot_joint(times=[0.05, 0.1, 0.2, 0.3])
plt.show()